In [2]:
import torch
import transformers
import bitsandbytes
import faiss

print("GPU 정상 인식:", torch.cuda.is_available())
print("FAISS 가속 모듈 로드 성공!")

GPU 정상 인식: True
FAISS 가속 모듈 로드 성공!


In [1]:
# 1. 오류가 나던 faiss-gpu 대신 Windows/CUDA12에 호환되는 공식 휠 주소로 명시적 설치
%pip install faiss-gpu-cu12 --index-url https://pypi.org

# 2. 나머지 HuggingFace 로컬 LLM 구동용 최적화 라이브러리 설치
%pip install langchain-huggingface sentence-transformers bitsandbytes accelerate


Looking in indexes: https://pypi.org
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement faiss-gpu-cu12 (from versions: none)
ERROR: No matching distribution found for faiss-gpu-cu12


  Using cached langchain_huggingface-1.2.2-py3-none-any.whl.metadata (4.0 kB)
  Using cached sentence_transformers-5.5.0-py3-none-any.whl.metadata (18 kB)
Using cached langchain_huggingface-1.2.2-py3-none-any.whl (31 kB)
Using cached sentence_transformers-5.5.0-py3-none-any.whl (588 kB)

   ---------------------------------------- 0/2 [langchain-huggingface]
   -------------------- ------------------- 1/2 [sentence-transformers]
   -------------------- ------------------- 1/2 [sentence-transformers]
   -------------------- ------------------- 1/2 [sentence-transformers]
   -------------------- ------------------- 1/2 [sentence-transformers]
   -------------------- ------------------- 1/2 [sentence-transformers]
   -------------------- ------------------- 1/2 [sentence-transformers]
   -------------------- ------------------- 1/2 [sentence-transformers]
   -------------------- ------------------- 1/2 [sentence-transformers]
   -------------------- ------------------- 1/2 [sentence-trans

In [1]:
import os
import torch
from typing import List, Dict, Any
from datasets import Dataset
from ranx import Qrels, Run, evaluate

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.llms import HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig

# ==========================================
# 1. 환경 및 하드웨어 가속 설정
# ==========================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"현재 사용 중인 하드웨어 장치: {device}")

# ==========================================
# 2. 가상의 RFP 테스트 데이터셋 구성
# ==========================================
# 실제 서비스 시에는 전처리한 문서 조각들을 여기에 매핑하세요.
mock_rfp_chunks = [
    Document(page_content="제5조(보안 관리): 본 사업의 개인정보 저장 시 AES-256 알고리즘으로 양방향 암호화 조치해야 하며, 비밀번호는 SHA-256 단방향 해시 처리한다.", metadata={"chunk_id": "chunk_001"}),
    Document(page_content="제12조(품질 보증): 시스템 가동 후 무상 유지보수 기간은 12개월로 하며, 월간 가동률은 99.9% 이상을 상시 유지하여야 한다.", metadata={"chunk_id": "chunk_002"}),
    Document(page_content="제23조(데이터 백업): 운영 데이터베이스 백업은 매일 자정(00:00)에 증분 백업을 수행하며, 백업본은 최소 3개월간 보관한다.", metadata={"chunk_id": "chunk_003"})
]

# Retriever 검증을 위한 평가 골든셋 (질문 - 정답 청크 ID 매핑)
test_dataset = [
    {
        "query_id": "q_1",
        "query": "비밀번호 암호화 표준 알고리즘 규칙이 어떻게 되나요?",
        "ground_truth_ids": ["chunk_001"]
    },
    {
        "query_id": "q_2",
        "query": "시스템 가동률 조건과 무상 유지보수 기간을 알려주세요.",
        "ground_truth_ids": ["chunk_002"]
    }
]

# ==========================================
# 3. 임베딩 모델 로드 및 FAISS Vector DB 빌드
# ==========================================
print("\n[1/4] 한국어 특화 'nlpai-lab/KURE-v1' 임베딩 모델 로드 중...")
embedding_model_name = "nlpai-lab/KURE-v1"
embeddings = HuggingFaceEmbeddings(
    model_name=embedding_model_name,
    model_kwargs={'device': device},
    encode_kwargs={'normalize_embeddings': True}
)

print("[2/4] FAISS Vector DB 인덱스 생성 중...")
vectorstore = FAISS.from_documents(mock_rfp_chunks, embeddings)
# 1차 후보군을 5개 추출하는 기본 Retriever 선언
faiss_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# ==========================================
# 4. Qwen2-7B-Instruct 4비트 양자화 로드 (코랩 최적화)
# ==========================================
print("[3/4] Qwen2-7B-Instruct 모델 양자화 로드 중 (시간이 다소 소요될 수 있습니다)...")
llm_model_id = "Qwen/Qwen2-7B-Instruct"

# 코랩 무료 T4 VRAM(16GB) 환경에서 안정적으로 구동하기 위한 4비트 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(llm_model_id)
model = AutoModelForCausalLM.from_pretrained(
    llm_model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.1
)
llm = HuggingFacePipeline(pipeline=pipe)

# ==========================================
# 5. Ranx 기반 정량 평가 모듈 정의 (함수화)
# ==========================================
def evaluate_retriever_performance(
    test_data: List[Dict[str, Any]],
    retriever: Any,
    metrics: List[str] = ["recall@1", "recall@3", "recall@5", "mrr", "ndcg@5"]
) -> Dict[str, float]:
    """
    FAISS Retriever 파이프라인의 검색 순위 퀄리티를 Ranx로 정량 평가합니다.
    """
    qrels_dict = {}
    run_dict = {}
    
    print("\n🚀 Ranx 평가를 위한 Retriever 추론 및 스코어 매핑 시작...")
    
    for idx, item in enumerate(test_data):
        q_id = item.get("query_id")
        query_text = item.get("query")
        gt_ids = item.get("ground_truth_ids")
        
        # 실제 정답 매핑
        qrels_dict[q_id] = {str(gt_id): 1 for gt_id in gt_ids}
        
        # FAISS 유사도 검색 수행
        retrieved_docs = retriever.get_relevant_documents(query_text)
        
        # Ranx 데이터 포맷에 맞춰 문서 점수 산출
        query_run_results = {}
        for rank, doc in enumerate(retrieved_docs):
            chunk_id = str(doc.metadata.get("chunk_id", f"unknown_{rank}"))
            # FAISS 유사도 스코어가 명시되지 않은 경우 순위 기반 패널티 스코어 부여
            score = doc.metadata.get("relevance_score", 1.0 / (rank + 1))
            query_run_results[chunk_id] = score
            
        run_dict[q_id] = query_run_results

    # Ranx 연산 실행
    qrels = Qrels(qrels_dict)
    run = Run(run_dict)
    results = evaluate(qrels, run, metrics)
    
    return dict(results) if isinstance(results, dict) else {metrics[0]: results}

# ==========================================
# 6. 전체 시스템 실행 및 최종 검증
# ==========================================
# 1) Ranx를 활용한 Retriever 검색 성능 지표 측정
eval_scores = evaluate_retriever_performance(
    test_data=test_dataset,
    retriever=faiss_retriever,
    metrics=["recall@1", "recall@3", "mrr"]
)

print("\n=============================================")
print("📊 [최종 결과] FAISS + KURE-v1 검색 성능 지표")
print("=============================================")
for metric, score in eval_scores.items():
    print(f"📌 {metric.upper().ljust(10)} : {score:.4f}")

# 2) 실제 Qwen2 LLM 연동 테스트 (샘플 1번 쿼리 적용)
print("\n=============================================")
print("🤖 Qwen2-7B-Instruct 최종 답변 생성 테스트")
print("=============================================")

sample_query = test_dataset[0]["query"]
search_docs = faiss_retriever.get_relevant_documents(sample_query)
context_str = "\n".join([d.page_content for d in search_docs])

prompt = ChatPromptTemplate.from_template("""
<|im_start|>system
당신은 사내 RFP 전문 분석가입니다. 주어진 참고 문서를 바탕으로 질문에 사실대로 요약 답변하세요.
<|im_end|>
<|im_start|>user
[참고 문서]
{context}

[질문]
{question}
<|im_end|>
<|im_start|>assistant
""")

chain = prompt | llm | StrOutputParser()
ai_response = chain.invoke({"context": context_str, "question": sample_query})

print(f"질문: {sample_query}")
print(f"답변:\n{ai_response}")

현재 사용 중인 하드웨어 장치: cuda

[1/4] 한국어 특화 'nlpai-lab/KURE-v1' 임베딩 모델 로드 중...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

d:\codeit\RAG\RFP-RAG-Extractor\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sooqw\.cache\huggingface\hub\models--nlpai-lab--KURE-v1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

#### **(위 코드) 에러가 발생한 원인 3**

이 에러는 4비트 양자화 모델을 불러올 때 사용 중인 그래픽 카드(GPU)의 전용 메모리(VRAM) 용량이 부족하여 모델 가중치가 RAM(CPU)이나 디스크로 튕겨 나갔기 때문에 발생하는 Hugging Face transformers의 안전 차단 기능입니다.4비트로 압축하더라도 LLM의 크기와 프롬프트 작동에 필요한 최소 VRAM 임계치를 넘지 못해 발생한 현상입니다.

이 문제를 우회하고 강제로 실행할 수 있는 2가지 해결책을 제안합니다.

#방법 1: CPU 오프로딩 허용 (가장 확실한 임시 조치)

부족한 GPU 메모리를 보완하기 위해 모델의 일부 레이어를 시스템 메모리(RAM)에 걸쳐놓고 구동하도록 설정을 명시해 줍니다.
코드를 아래와 같이 수정하세요. quantization_config 내부에 오프로딩 활성화 인자를 추가해야 합니다.

단점: VRAM 영역을 벗어나 일반 RAM(CPU) 영역을 오가며 계산하기 때문에 텍스트 생성 및 Ranx 정량 평가 속도가 현저히 느려집니다.

In [ ]:
from transformers import BitsAndBytesConfig, AutoModelForCausalLM, AutoTokenizer

# 1. 4비트 설정에 CPU 오프로드 옵션 강제 추가
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    llm_int8_enable_fp32_cpu_offload=True  # <- 이 부분을 추가합니다
)

tokenizer = AutoTokenizer.from_pretrained(llm_model_id)

# 2. 모델 로드 시 CPU 및 디스크 분할 배치를 승인
model = AutoModelForCausalLM.from_pretrained(
    llm_model_id,
    quantization_config=bnb_config,
    device_map="auto" # 'auto' 상태에서 오프로드가 연동됩니다
)


#방법 2: 장치 명시 및 데이터 타입 다운그레이드 (속도 유지형 조치)

메모리가 아슬아슬하게 부족한 경우, device_map="auto"가 셰어링을 계산하다가 예외를 던진 것일 수 있습니다. 디바이스를 GPU 0번으로 명확하게 지정하고 가속 연산 정밀도를 강제 조정해 봅니다.

만약 이 설정을 적용했는데 Out of Memory (OOM) 에러가 즉시 터진다면, 물리적인 VRAM 한계에 도달한 것이므로 방법 1을 적용하거나 더 작은 체급의 LLM 모델(예: 7B에서 3B나 1B 체급으로 변경)을 선택해야 합니다.

우선 코드를 방법 1 형태로 수정한 뒤 Ranx 정량 평가 셀이 끝까지 통과하는지 확인해 보세요.
만약 수정한 후에도 다른 문법적 거부 반응이나 속도가 지나치게 느려져 멈춘 것처럼 보인다면, 현재 불러오고자 하는 LLM 모델명(llm_model_id)과 그래픽 카드의 VRAM 용량(예: RTX 4060 8GB 등)을 공유해 주세요. 최적의 모델 체급을 다시 세팅해 드리겠습니다.

In [ ]:
import torch

model = AutoModelForCausalLM.from_pretrained(
    llm_model_id,
    quantization_config=bnb_config,
    device_map="cuda:0",               # <- auto 대신 첫 번째 GPU로 직행 고정
    torch_dtype=torch.float16          # <- bfloat16 지원이 약한 환경일 시 float16으로 대체
)